# Notebook 03: Label compounds

The compounds whose family was checked by hand are labeled. Every other compound stays unlabeled and is used later by the semi-supervised model.

### Load the compounds
`compounds.csv` is read with its extra spaces removed. Only the InChIKey and the name are needed here.

In [1]:
import pandas as pd

compounds = pd.read_csv("../data/processed/compounds.csv", skipinitialspace=True)
compounds.columns = compounds.columns.str.strip()
compounds["inchikey"] = compounds.inchikey.str.strip()
compounds["name"] = compounds.name.str.strip().str.replace("\xa0", " ")
compounds = compounds[["inchikey", "name"]]
compounds.shape

(1570, 2)

### Load the checked list
`known_compounds.csv` holds the compounds checked by hand: InChIKey, name, family, class, and whether the row is verified.

In [2]:
known = pd.read_csv("../data/labels/known_compounds.csv")
known.head()

,inchikey,name,family,class,verified
0,KWTSXDURSIMDCE-UHFFFAOYSA-N,Amphetamine,amphetamine-type,stimulant,yes
1,MYWUZJCMWCOHBA-UHFFFAOYSA-N,Methamphetamine,amphetamine-type,stimulant,yes
2,DHHVAGZRUROJKS-UHFFFAOYSA-N,Phentermine,amphetamine-type,stimulant,yes
3,ZPUCINDJVBIVPJ-UHFFFAOYSA-N,Cocaine,stimulant-other,stimulant,yes
4,DUGOZIWVEXMGBE-UHFFFAOYSA-N,Dexmethylphenidate,stimulant-other,stimulant,yes


### Load the family-to-class table
Each family belongs to exactly one class.

In [3]:
family_to_class = pd.read_csv("../data/labels/family_to_class.csv")
family_to_class

,family,class
0,amphetamine-type,stimulant
1,cathinone,stimulant
2,pyrrolidinophenone,stimulant
3,stimulant-other,stimulant
4,fentanyl-type,opioid
5,nitazene,opioid
6,morphinan,opioid
7,opioid-other,opioid
8,indole-ketone-SC,cannabinoid
9,carboxamide-ester-SC,cannabinoid


### Keep verified rows
Only rows with `verified = yes` are used as labels.

In [4]:
known = known[known.verified == "yes"]
len(known)

637

### Check that every compound exists
Each InChIKey in the list must be in the dataset.

In [5]:
missing = ~known.inchikey.isin(compounds.inchikey)
print("InChIKeys not in the dataset:", missing.sum())
assert not missing.any()

InChIKeys not in the dataset: 0


### Remove compounds listed twice
A compound listed twice could carry two different families. Both rows are dropped and the compound stays unlabeled.

In [6]:
twice = known.inchikey.duplicated(keep=False)
print("rows dropped:", twice.sum())
known = known[~twice]

rows dropped: 0


### Check that every family has a class

In [7]:
lookup = dict(zip(family_to_class.family, family_to_class["class"]))

no_class = ~known.family.isin(lookup)
print("families without a class:", no_class.sum())
assert not no_class.any()

families without a class: 0


### Check that each class matches its family
The class in the list must equal the class given by the table.

In [8]:
wrong = known["class"] != known.family.map(lookup)
print("rows with a different class:", wrong.sum())
assert not wrong.any()

rows with a different class: 0


### Count compounds per family

In [9]:
counts = known.family.value_counts().reindex(family_to_class.family, fill_value=0)
counts

family
amphetamine-type              33
cathinone                     38
pyrrolidinophenone            32
stimulant-other               31
fentanyl-type                 51
nitazene                      32
morphinan                     24
opioid-other                  41
indole-ketone-SC              32
carboxamide-ester-SC          44
cannabinoid-other             15
tryptamine                    36
phenethylamine-psychedelic    30
psychedelic-other             17
arylcyclohexylamine           30
benzodiazepine                37
depressant-other              24
steroid                       16
other-medicine                74
Name: count, dtype: int64

### Families below the target
The target is 30 compounds per family. Smaller families are listed, and the run continues.

In [10]:
MIN_PER_FAMILY = 30
counts[counts < MIN_PER_FAMILY]

family
morphinan            24
cannabinoid-other    15
psychedelic-other    17
depressant-other     24
steroid              16
Name: count, dtype: int64

### Label the compounds
Listed compounds get their family and class and the status `known`. All other compounds get no label and the status `unlabeled`.

In [11]:
labels = compounds.merge(known[["inchikey", "family", "class"]], on="inchikey", how="left")
labels["status"] = labels.family.notna().map({True: "known", False: "unlabeled"})
labels.head()

,inchikey,name,family,class,status
0,LQZHCIPFXVLYCQ-UHFFFAOYSA-N,4-Phenylmethamphetamine,amphetamine-type,stimulant,known
1,VQECHRQHFMUVRS-UHFFFAOYSA-N,"1-(1,3-Diphenylpropan-2-yl)pyrrolidine",NaN,NaN,unlabeled
2,KITIJFNTLBMYQD-UHFFFAOYSA-N,"1-(2,5-Dimethoxybenzyl)piperazine",NaN,NaN,unlabeled
3,UFRAAIGOXGTHQX-UHFFFAOYSA-N,1-(2-Chloro-3-fluorophenyl)piperazine,stimulant-other,stimulant,known
4,VNZLQLYBRIOLFZ-UHFFFAOYSA-N,1-(2-Methoxyphenyl)piperazine,stimulant-other,stimulant,known


### Check and save
Every compound appears once. The table is saved for notebook 04.

In [12]:
assert len(labels) == len(compounds)
assert labels.inchikey.is_unique

labels.to_csv("../data/labels/labels.csv", index=False)

### Summary

In [13]:
labels.status.value_counts()

status
unlabeled    933
known        637
Name: count, dtype: int64

In [14]:
labels["class"].value_counts()

class
opioid          148
stimulant       134
cannabinoid      91
other            90
psychedelic      83
depressant       61
dissociative     30
Name: count, dtype: int64